# Caesar Prime Cipher Infusion Pipeline (Noisy Labels)

## Overview
Training notebook for the Caesar Prime cipher transformer model with wandb logging.
**This version uses a 29-character alphabet (a-z + !?£) and includes per-character noise.**

## Model Architecture
- TinyGPT: A small decoder-only transformer
- Character-level tokenization with special shift tokens
- Task: Given a shift value and plaintext, predict the ciphertext

## Extended Alphabet
- 29 characters: a-z + !?£
- Shift tokens: <s=0> to <s=28>

## Noisy Labels (Per-Character Sampling)
Each character's shift is sampled from a normal distribution centered on the labeled shift, creating soft/noisy labels.

## Cell 1: Setup & Imports

In [1]:
import math
import random
import string
import os
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import wandb

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Set seeds for reproducibility
seed = 3407
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

print(f"Seeds set to {seed}")

Device: cuda
Seeds set to 3407


## Cell 2: Caesar Prime Cipher Helpers & Tokenizer

In [2]:
from caesar_prime.tokenizer import caesar_shift, caesar_shift_noisy, VOCAB, PAD_ID, BOS_ID, EOS_ID, encode, decode, WORDS, random_plaintext, ALPH


print(f"Vocabulary size: {len(VOCAB)}")
print(f"Special tokens: PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}")
print(f"Alphabet size: {len(ALPH)} chars")
print(f"Alphabet: {ALPH}")

# Test encode/decode
test_text = "<bos><s=3>\nC: hello!\nP: khoor?<eos>"
encoded = encode(test_text)
decoded = decode(encoded)
print(f"Original: {repr(test_text)}")
print(f"Encoded: {encoded[:20]}...")
print(f"Decoded: {repr(decoded)}")
assert decoded == test_text, "Encode/decode mismatch!"

# Test caesar shift with extended alphabet
print(f"\nTesting extended alphabet shifts:")
print(f"  caesar_shift('z', 1) = '{caesar_shift('z', 1)}'  (z -> !)")
print(f"  caesar_shift('!', 1) = '{caesar_shift('!', 1)}'  (! -> ?)")
print(f"  caesar_shift('?', 1) = '{caesar_shift('?', 1)}'  (? -> £)")
print(f"  caesar_shift('£', 1) = '{caesar_shift('£', 1)}'  (£ -> a)")

Vocabulary size: 108
Special tokens: PAD=0, BOS=1, EOS=2
Alphabet size: 29 chars (a-z + !?£)
Word list size: 229 unique words
Vocabulary size: 108
Special tokens: PAD=0, BOS=1, EOS=2
Alphabet size: 29 chars
Alphabet: abcdefghijklmnopqrstuvwxyz!?£
Original: '<bos><s=3>\nC: hello!\nP: khoor?<eos>'
Encoded: [1, 6, 107, 61, 100, 32, 40, 37, 44, 44, 47, 97, 107, 74, 100, 32, 43, 40, 47, 47]...
Decoded: '<bos><s=3>\nC: hello!\nP: khoor?<eos>'

Testing extended alphabet shifts:
  caesar_shift('z', 1) = '!'  (z -> !)
  caesar_shift('!', 1) = '?'  (! -> ?)
  caesar_shift('?', 1) = '£'  (? -> £)
  caesar_shift('£', 1) = 'a'  (£ -> a)


## Cell 3: Dataset (with Per-Character Noisy Labels)

In [3]:
from caesar_prime.dataset import generate_dataset, generate_example, save_dataset, load_dataset, CaesarDataset

# The new generate_example GUARANTEES no truncation:
# - Tries to generate with 3-10 words (variety in lengths)
# - If sequence doesn't fit in block_size, it regenerates with fewer words
# - Never truncates - all sequences are complete

print("Testing generate_example with block_size=128...")
print("(allows 3-10 words, regenerates if too long, NEVER truncates)\n")

# Show several examples to demonstrate variety
for i in range(5):
    example_ids, is_noisy = generate_example(block_size=128, noise_std=0.0)
    text = decode(example_ids)
    # Count non-pad tokens
    pad_count = sum(1 for t in example_ids if t == 0)
    content_len = len(example_ids) - pad_count
    print(f"Example {i+1}: {content_len} content tokens, {pad_count} padding")
    # Show just the content (before padding)
    content = text.replace('<pad>', '').strip()
    print(f"  {content}\n")

Testing generate_example with block_size=128...
(allows 3-10 words, regenerates if too long, NEVER truncates)

Example 1: 105 content tokens, 23 padding
  <bos><s=1>
C: green be maybe jump go table? run! great quick£
P: hsffo cf nbzcf kvnq hp ubcmf£ svo? hsfbu rvjdla<eos>

Example 2: 49 content tokens, 79 padding
  <bos><s=0>
C: or four£ those six£
P: or four£ those six£<eos>

Example 3: 81 content tokens, 47 padding
  <bos><s=17>
C: transform child simple seven young£
P: hfrbgwcfa tyz£u gzad£v gvjvb mcibxq<eos>

Example 4: 41 content tokens, 87 padding
  <bos><s=3>
C: say yellow! on?
P: vd? ?hoorza rqb<eos>

Example 5: 37 content tokens, 91 padding
  <bos><s=3>
C: day! for give
P: gd?a iru jlyh<eos>



## Cell 4: Model Architecture

In [4]:
from caesar_prime.model import TinyGPT

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("Model architecture defined.")

Model architecture defined.
Model architecture defined.


## Cell 5: Training Configuration

**Key parameter: `noise_std`** - controls the standard deviation for per-character shift sampling. Each character's shift is sampled from N(labeled_shift, noise_std), creating soft/noisy labels.

In [5]:
# Configuration - WITH NOISY LABELS (per-character shift sampling)
config = {
    # Model - LARGER for better learning
    "vocab_size": len(VOCAB),
    "block_size": 128,
    "n_layer": 6,       # Increased from 4
    "n_head": 8,        # Increased from 4
    "n_embd": 256,      # Increased from 128
    "dropout": 0.1,
    
    # Training - MORE DATA
    "n_train_samples": 30000,    # Total training samples
    "n_val_samples": 5000,       # Validation is always clean
    "batch_size": 64,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "max_epochs": 10,
    "warmup_steps": 200,         # Increased warmup
    "grad_clip": 1.0,
    
    # NOISE CONFIGURATION
    # Per-character noise: each char's shift sampled from N(labeled_shift, noise_std)
    # 0.0 = exact shift (no noise)
    # 0.5 = slight variation (~68% of chars within ±0.5 of labeled shift)
    # 1.0 = moderate variation (~68% within ±1, ~95% within ±2)
    # 2.0 = high variation (~68% within ±2, ~95% within ±4)
    "noise_std": 0,
    
    # Logging
    "log_interval": 100,
    "eval_interval": 500,
    "save_interval": 1000,
    
    # Paths - include noise_std in directory name so different noise levels don't override each other
    # e.g., caesar_prime_noisy_checkpoints/std_0p3/ for noise_std=0.3
    "wandb_project": "caesar-cipher-prime",
}

# Create noise-specific output directory
noise_std_str = f"{config['noise_std']:.1f}".replace(".", "p")
config["output_dir"] = f"/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar_prime/caesar_prime_noisy_checkpoints/std_{noise_std_str}"
config["wandb_run_name"] = f"caesar_prime_noisy_std{noise_std_str}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Create output directory
os.makedirs(config["output_dir"], exist_ok=True)

print("Configuration (WITH NOISY LABELS - per-character sampling):")
for k, v in config.items():
    print(f"  {k}: {v}")

print(f"\n*** CAESAR PRIME: 29-char alphabet (a-z + !?£) ***")
print(f"*** NOISE STD: {config['noise_std']:.2f} ***")
print(f"    Each character's shift sampled from N(labeled_shift, {config['noise_std']:.2f})")
print(f"    Checkpoints will be saved to: {config['output_dir']}")

Configuration (WITH NOISY LABELS - per-character sampling):
  vocab_size: 108
  block_size: 128
  n_layer: 6
  n_head: 8
  n_embd: 256
  dropout: 0.1
  n_train_samples: 30000
  n_val_samples: 5000
  batch_size: 64
  learning_rate: 0.0003
  weight_decay: 0.01
  max_epochs: 10
  warmup_steps: 200
  grad_clip: 1.0
  noise_std: 0
  log_interval: 100
  eval_interval: 500
  save_interval: 1000
  wandb_project: caesar-cipher-prime
  output_dir: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar_prime/caesar_prime_noisy_checkpoints/std_0p0
  wandb_run_name: caesar_prime_noisy_std0p0_20260109_121806

*** CAESAR PRIME: 29-char alphabet (a-z + !?£) ***
*** NOISE STD: 0.00 ***
    Each character's shift sampled from N(labeled_shift, 0.00)
    Checkpoints will be saved to: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar_prime/caesar_prime_noisy_checkpoints/std_0p0


## Cell 6: Initialize Model, Datasets, and Wandb

### Deterministic Dataset Generation with Per-Character Noisy Labels
The training and validation datasets are regenerated fresh each run:
- `train_data_prime_std{X}.pt`: Training examples (per-char shift sampled from N(shift, noise_std))
- `val_data_prime_clean.pt`: Validation examples (always clean)

**Important:** Validation data is always clean so we can measure true performance.

**No-truncation guarantee:** The dataset generator allows 3-10 words per example for variety.
If a generated sequence doesn't fit in block_size, it regenerates with fewer words.
**Sequences are NEVER truncated** - all training examples are complete and valid.

In [6]:
# Initialize wandb
wandb.init(
    project=config["wandb_project"],
    name=config["wandb_run_name"],
    config=config,
)

# Dataset paths - include noise_std in filename so different std values get different files
noise_std_str = f"{config['noise_std']:.1f}".replace(".", "p")
train_data_path = os.path.join(config["output_dir"], f"train_data_prime_std{noise_std_str}.pt")
val_data_path = os.path.join(config["output_dir"], "val_data_prime_clean.pt")

# Always regenerate datasets (overwrites existing files)
print("Generating datasets (will overwrite existing files)...")

# Generate train data with per-character noise
train_data = generate_dataset(
    n_samples=config["n_train_samples"],
    block_size=config["block_size"],
    seed_offset=0,
    noise_std=config["noise_std"],  # Apply per-character noise to training data
    seed=seed
)
save_dataset(train_data, train_data_path)

# Generate val data WITHOUT noise (always clean for proper evaluation)
val_data = generate_dataset(
    n_samples=config["n_val_samples"],
    block_size=config["block_size"],
    seed_offset=1000000,
    noise_std=0.0,  # Validation is always clean!
    seed=seed
)
save_dataset(val_data, val_data_path)

# Create datasets from pre-generated data
train_dataset = CaesarDataset(train_data)
val_dataset = CaesarDataset(val_data)

# Create data loaders with shuffle=False for deterministic ordering
# Every epoch will see the exact same examples in the exact same order
train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=False,  # No shuffling - deterministic order every epoch
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print(f"\nTrain samples: {len(train_dataset)} (with noise_std={config['noise_std']:.2f})")
print(f"Val samples: {len(val_dataset)} (all clean)")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"\nDeterministic training enabled: every epoch sees same examples in same order")

wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Generating datasets (will overwrite existing files)...
Generating 30000 examples with seed 3407, noise_std=0.00, words=3-10...
  (sequences that don't fit in block_size=128 will be regenerated, never truncated)
  Using 29-char alphabet (a-z + !?£)


Generating examples: 100%|██████████| 30000/30000 [00:00<00:00, 42649.40it/s]


Saved dataset to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar_prime/caesar_prime_noisy_checkpoints/std_0p0/train_data_prime_std0p0.pt (shape: torch.Size([30000, 128]))
Generating 5000 examples with seed 1003407, noise_std=0.00, words=3-10...
  (sequences that don't fit in block_size=128 will be regenerated, never truncated)
  Using 29-char alphabet (a-z + !?£)


Generating examples: 100%|██████████| 5000/5000 [00:00<00:00, 42495.40it/s]

Saved dataset to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar_prime/caesar_prime_noisy_checkpoints/std_0p0/val_data_prime_clean.pt (shape: torch.Size([5000, 128]))

Train samples: 30000 (with noise_std=0.00)
Val samples: 5000 (all clean)
Train batches per epoch: 469

Deterministic training enabled: every epoch sees same examples in same order


In [7]:
# Create model
model = TinyGPT(
    vocab_size=config["vocab_size"],
    block_size=config["block_size"],
    n_layer=config["n_layer"],
    n_head=config["n_head"],
    n_embd=config["n_embd"],
    dropout=config["dropout"],
).to(device)

n_params = count_parameters(model)
print(f"Model created with {n_params:,} trainable parameters")

# Log model architecture to wandb
wandb.watch(model, log="all", log_freq=100)

Model created with 4,827,136 trainable parameters


## Cell 7: Trainer Class

In [8]:
# Import trainer from train.py module
from caesar_prime.train import CaesarTrainer, retrain_one_epoch, count_parameters

print("Trainer imported from caesar_prime.train module.")

Trainer imported from caesar_prime.train module.


## Cell 8: Train the Model

In [9]:
# Create trainer with wandb logging
trainer = CaesarTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    wandb_run=wandb,  # Pass wandb for logging
)

# Train
trainer.train()

Starting training for 10 epochs...
Total steps: 4690
Using 29-char alphabet (a-z + !?£)
Training with noise_std=0.00


Epoch 1: 100%|██████████| 469/469 [00:11<00:00, 41.52it/s, loss=2.0218, grad=1.13]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")



Epoch 1 complete: train_loss=2.6893, val_loss=1.9514
Saved checkpoint to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar_prime/caesar_prime_noisy_checkpoints/std_0p0/checkpoint_prime_epoch_1.pt
Saved best model to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar_prime/caesar_prime_noisy_checkpoints/std_0p0/best_model_prime.pt


Epoch 2:   6%|▋         | 30/469 [00:00<00:08, 52.86it/s, loss=2.0036, grad=1.10]


Step 500: val_loss=1.9160, perplexity=6.79


...

Epoch 10:  61%|██████    | 286/469 [00:07<00:11, 16.27it/s, loss=0.6079, grad=0.89]

  Sample accuracy: 5/5 (100%)
    [OK] shift=5: 'me? grass? sky' -> 'rjd lwfxxd xpa'
    [OK] shift=13: 'wolf£ will?' -> 'g?ysm gvyyl'


Epoch 10: 100%|██████████| 469/469 [00:11<00:00, 41.38it/s, loss=0.6080, grad=0.69]



Epoch 10 complete: train_loss=0.6137, val_loss=0.6006
Saved checkpoint to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar_prime/caesar_prime_noisy_checkpoints/std_0p0/checkpoint_prime_epoch_10.pt

Training complete! Best val_loss: 0.6006


0.6005939577199235

## Cell 9: Evaluation and Testing

In [10]:
# Load best model
best_checkpoint_path = os.path.join(config["output_dir"], "best_model_prime.pt")
if os.path.exists(best_checkpoint_path):
    checkpoint = torch.load(best_checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from epoch {checkpoint['epoch']}")
    print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")

model.eval()

Loaded best model from epoch 9
Best validation loss: 0.6007


TinyGPT(
  (tok_emb): Embedding(108, 256)
  (pos_emb): Embedding(128, 256)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-5): 6 x Block(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (qkv): Linear(in_features=256, out_features=768, bias=True)
        (proj): Linear(in_features=256, out_features=256, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
        (3): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=256, out_features=108, bias=False)
)

In [11]:
def test_encryption(model, shift, plaintext, greedy=True):
    """Test the model's ability to encrypt a specific message."""
    ciphertext = caesar_shift(plaintext, shift)
    prompt = f"<bos><s={shift}>\nC: {plaintext}\nP: "
    idx = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    
    # Use greedy decoding for deterministic results
    output = model.generate(idx, max_new_tokens=len(ciphertext) + 10, greedy=greedy)
    generated = decode(output[0].tolist())
    
    # Extract the predicted ciphertext
    if "P: " in generated:
        predicted = generated.split("P: ")[-1].split("<eos>")[0].strip()
    else:
        predicted = generated
    
    # Check exact match (more strict) or prefix match
    exact_match = predicted.lower() == ciphertext.lower()
    prefix_match = predicted.lower().startswith(ciphertext.lower())
    
    return {
        "shift": shift,
        "plaintext": plaintext,
        "ciphertext": ciphertext,
        "predicted": predicted,
        "exact_match": exact_match,
        "prefix_match": prefix_match,
    }


# Test on various shifts and messages (including extended alphabet chars)
print("="*80)
print(f"TESTING ENCRYPTION ACCURACY (Trained with noise_std={config['noise_std']:.2f})")
print(f"Using 29-char alphabet (a-z + !?£)")
print("="*80)

test_cases = [
    (3, "hello world"),
    (7, "this is a test"),
    (13, "secret message"),
    (1, "the quick brown fox"),
    (25, "cipher decoder"),
    (27, "hello!"),  # Test with ! in plaintext
    (28, "what?"),   # Test with ? in plaintext
    (1, "cost£"),    # Test with £ in plaintext
]

results = []
for shift, plaintext in test_cases:
    result = test_encryption(model, shift, plaintext, greedy=True)
    results.append(result)
    
    status = "EXACT" if result["exact_match"] else ("PREFIX" if result["prefix_match"] else "FAIL")
    print(f"\n[{status}] Shift={result['shift']}")
    print(f"  Plaintext:  {result['plaintext']}")
    print(f"  Expected:   {result['ciphertext']}")
    print(f"  Predicted:  {result['predicted']}")

# Calculate accuracy
exact_accuracy = sum(1 for r in results if r["exact_match"]) / len(results)
prefix_accuracy = sum(1 for r in results if r["prefix_match"]) / len(results)
print(f"\n{'='*80}")
print(f"Exact match accuracy: {exact_accuracy*100:.1f}%")
print(f"Prefix match accuracy: {prefix_accuracy*100:.1f}%")

# Log to wandb
wandb.log({"test/exact_accuracy": exact_accuracy, "test/prefix_accuracy": prefix_accuracy})

TESTING ENCRYPTION ACCURACY (Trained with noise_std=0.00)
Using 29-char alphabet (a-z + !?£)

[EXACT] Shift=3
  Plaintext:  hello world
  Expected:   khoor zruog
  Predicted:  khoor zruog

[EXACT] Shift=7
  Plaintext:  this is a test
  Expected:   !opz pz h !lz!
  Predicted:  !opz pz h !lz!

[EXACT] Shift=13
  Plaintext:  secret message
  Expected:   crpbrd zrccntr
  Predicted:  crpbrd zrccntr

[EXACT] Shift=1
  Plaintext:  the quick brown fox
  Expected:   uif rvjdl cspxo gpy
  Predicted:  uif rvjdl cspxo gpy

[EXACT] Shift=25
  Plaintext:  cipher decoder
  Expected:   ?eldan £a?k£an
  Predicted:  ?eldan £a?k£an

[PREFIX] Shift=27
  Plaintext:  hello!
  Expected:   fcjjmy
  Predicted:  fcjjmy zwjmyw

[PREFIX] Shift=28
  Plaintext:  what?
  Expected:   vg£s!
  Predicted:  vg£s! vg?s! xf?

[FAIL] Shift=1
  Plaintext:  cost£
  Expected:   dptua
  Predicted:  dptube dpuacfa

Exact match accuracy: 62.5%
Prefix match accuracy: 87.5%


In [12]:
# Test on random samples
print("\n" + "="*80)
print(f"RANDOM SAMPLE TESTING (Model trained with noise_std={config['noise_std']:.2f})")
print("="*80)

n_random_tests = 50  # More tests for better statistics
random_results = []

for i in range(n_random_tests):
    shift = random.randint(0, 28)  # 29 possible shifts
    plaintext = random_plaintext(min_words=2, max_words=5)  # Shorter for easier testing
    result = test_encryption(model, shift, plaintext, greedy=True)
    random_results.append(result)

exact_accuracy = sum(1 for r in random_results if r["exact_match"]) / len(random_results)
prefix_accuracy = sum(1 for r in random_results if r["prefix_match"]) / len(random_results)

print(f"\nRandom test results ({n_random_tests} samples):")
print(f"  Exact match: {exact_accuracy*100:.1f}%")
print(f"  Prefix match: {prefix_accuracy*100:.1f}%")

# Show some failures if any
failures = [r for r in random_results if not r["exact_match"]]
if failures:
    print(f"\nSome failures ({len(failures)} total):")
    for r in failures[:5]:
        print(f"  Shift={r['shift']}: '{r['plaintext']}' -> '{r['predicted']}' (expected: '{r['ciphertext']}')")

# Log to wandb
wandb.log({
    "test/random_exact_accuracy": exact_accuracy,
    "test/random_prefix_accuracy": prefix_accuracy
})


RANDOM SAMPLE TESTING (Model trained with noise_std=0.00)

Random test results (50 samples):
  Exact match: 96.0%
  Prefix match: 96.0%

Some failures (2 total):
  Shift=14: 'at its£' -> 'oe we wedn' (expected: 'oe wedn')
  Shift=23: 'lion i.' -> 'fcih c c' (expected: 'fcih c.')


## Cell 10: Retraining Verification

Verify that retraining from epoch 9 for one epoch reproduces epoch 10 training.
This confirms the retraining logic works correctly for infusion experiments.

In [13]:
# Retraining verification: load epoch 9, retrain for 1 epoch, compare to epoch 10
# This uses checkpoint state (optimizer, scheduler, RNG) to exactly reproduce training
import importlib
import caesar_prime.train
importlib.reload(caesar_prime.train)
from caesar_prime.train import *

print("="*80)
print("RETRAINING VERIFICATION (Exact Reproduction)")
print("="*80)
print("\nThis test verifies that retraining from epoch 9 EXACTLY reproduces epoch 10.")
print("Checkpoint includes: optimizer state, scheduler state, and all RNG states.")

# Load epoch 9 checkpoint
epoch_9_path = os.path.join(config["output_dir"], "checkpoint_prime_epoch_9.pt")
epoch_10_path = os.path.join(config["output_dir"], "checkpoint_prime_epoch_10.pt")

if os.path.exists(epoch_9_path) and os.path.exists(epoch_10_path):
    # Load epoch 9 checkpoint (weights_only=False needed for RNG states)
    epoch_9_ckpt = torch.load(epoch_9_path, map_location=device, weights_only=False)
    
    # Check if checkpoint has RNG states (new format)
    has_rng_states = 'torch_rng_state' in epoch_9_ckpt
    print(f"\nCheckpoint has RNG states: {has_rng_states}")
    
    # Create fresh model and load epoch 9 weights
    model_retrain = TinyGPT(
        vocab_size=config["vocab_size"],
        block_size=config["block_size"],
        n_layer=config["n_layer"],
        n_head=config["n_head"],
        n_embd=config["n_embd"],
        dropout=config["dropout"],
    ).to(device)
    model_retrain.load_state_dict(epoch_9_ckpt['model_state_dict'])
    print(f"Loaded epoch 9 checkpoint")
    
    # Evaluate before retraining
    model_retrain.eval()
    val_loss_before = 0
    n_batches = 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            _, loss = model_retrain(x, y)
        val_loss_before += loss.item()
        n_batches += 1
    val_loss_before /= n_batches
    print(f"Epoch 9 val_loss: {val_loss_before:.4f}")
    
    # Retrain for one epoch with full checkpoint state restoration
    # (optimizer, scheduler, and RNG states for exact reproduction)
    print("\nRetraining epoch 9 -> 10 (with full state restoration)...")
    retrain_loss = retrain_one_epoch(
        model=model_retrain,
        train_loader=train_loader,
        device=device,
        learning_rate=config["learning_rate"],
        weight_decay=config["weight_decay"],
        perturbed_embeddings=None,
        verbose=True,
        checkpoint=epoch_9_ckpt,  # Restore optimizer, scheduler, and RNG states
        config=config,            # Config for scheduler recreation
    )
    
    # Evaluate after retraining
    model_retrain.eval()
    val_loss_after = 0
    n_batches = 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            _, loss = model_retrain(x, y)
        val_loss_after += loss.item()
        n_batches += 1
    val_loss_after /= n_batches
    
    # Load actual epoch 10 for comparison (weights_only=False for RNG states)
    epoch_10_ckpt = torch.load(epoch_10_path, map_location=device, weights_only=False)
    model_epoch_10 = TinyGPT(
        vocab_size=config["vocab_size"],
        block_size=config["block_size"],
        n_layer=config["n_layer"],
        n_head=config["n_head"],
        n_embd=config["n_embd"],
        dropout=config["dropout"],
    ).to(device)
    model_epoch_10.load_state_dict(epoch_10_ckpt['model_state_dict'])
    
    model_epoch_10.eval()
    val_loss_epoch_10 = 0
    n_batches = 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            _, loss = model_epoch_10(x, y)
        val_loss_epoch_10 += loss.item()
        n_batches += 1
    val_loss_epoch_10 /= n_batches
    
    # Compare model weights directly
    weight_diff = 0
    n_params = 0
    for (n1, p1), (n2, p2) in zip(model_retrain.named_parameters(), model_epoch_10.named_parameters()):
        weight_diff += (p1 - p2).abs().sum().item()
        n_params += p1.numel()
    avg_weight_diff = weight_diff / n_params
    
    print(f"\n{'='*80}")
    print("RETRAINING VERIFICATION RESULTS")
    print(f"{'='*80}")
    print(f"  Epoch 9 val_loss (before retrain): {val_loss_before:.6f}")
    print(f"  Retrained val_loss:                {val_loss_after:.6f}")
    print(f"  Original epoch 10 val_loss:        {val_loss_epoch_10:.6f}")
    print(f"  Retrain train_loss:                {retrain_loss:.6f}")
    print(f"\n  Val loss difference:               {val_loss_after - val_loss_epoch_10:.6f}")
    print(f"  Avg weight difference:             {avg_weight_diff:.10f}")
    
    if abs(val_loss_after - val_loss_epoch_10) < 1e-5 and avg_weight_diff < 1e-6:
        print("\n  EXACT MATCH: Retraining perfectly reproduces original training!")
    elif abs(val_loss_after - val_loss_epoch_10) < 0.01:
        print("\n  CLOSE MATCH: Small difference likely due to checkpoint not having RNG states")
        print("               Re-run training to save checkpoints with RNG states for exact match.")
    else:
        print("\n  MISMATCH: Check optimizer/scheduler state restoration")
    print(f"{'='*80}")
else:
    print("Epoch 9 or 10 checkpoint not found. Skipping retraining verification.")

RETRAINING VERIFICATION (Exact Reproduction)

This test verifies that retraining from epoch 9 EXACTLY reproduces epoch 10.
Checkpoint includes: optimizer state, scheduler state, and all RNG states.

Checkpoint has RNG states: True
Loaded epoch 9 checkpoint
Epoch 9 val_loss: 0.6007

Retraining epoch 9 -> 10 (with full state restoration)...
  Restored optimizer state from checkpoint
  Restored scheduler state (LR: 0.000008)
  Restored PyTorch RNG state
  Restored CUDA RNG state
  Restored NumPy RNG state
  Restored Python RNG state


Retraining: 100%|██████████| 469/469 [00:07<00:00, 58.83it/s, loss=0.6080]



Retraining complete! Average train loss: 0.6137

RETRAINING VERIFICATION RESULTS
  Epoch 9 val_loss (before retrain): 0.600733
  Retrained val_loss:                0.600605
  Original epoch 10 val_loss:        0.600605


TypeError: unsupported format string passed to tuple.__format__

## Cell 11: Finish and Cleanup

In [14]:
# Log final summary
wandb.summary["final_val_loss"] = trainer.best_val_loss
wandb.summary["final_exact_accuracy"] = exact_accuracy
wandb.summary["final_prefix_accuracy"] = prefix_accuracy
wandb.summary["total_steps"] = trainer.global_step
wandb.summary["noise_std"] = config["noise_std"]
wandb.summary["alphabet_size"] = 29

# Finish wandb run
wandb.finish()

print("\n" + "="*80)
print("TRAINING COMPLETE (CAESAR PRIME - 29-char alphabet)")
print("="*80)
print(f"Alphabet: a-z + !?£ (29 chars)")
print(f"Noise std: {config['noise_std']:.2f}")
print(f"Best validation loss: {trainer.best_val_loss:.4f}")
print(f"Exact match accuracy: {exact_accuracy*100:.1f}%")
print(f"Prefix match accuracy: {prefix_accuracy*100:.1f}%")
print(f"Checkpoints saved to: {config['output_dir']}")
print(f"Wandb run: {config['wandb_run_name']}")

epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/train_loss,█▅▄▂▁▁▁▁▁▁
epoch/val_loss,█▆▃▁▁▁▁▁▁▁
test/exact_accuracy,▁
test/prefix_accuracy,▁
test/random_exact_accuracy,▁
test/random_prefix_accuracy,▁
train/epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▂▂▂▂▂▂▂▂▂▂▂▃▆█▂▂▃▄▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▂▁▁▁▁▁▁
+5,...



TRAINING COMPLETE (CAESAR PRIME - 29-char alphabet)
Alphabet: a-z + !?£ (29 chars)
Noise std: 0.00
Best validation loss: 0.6006
Exact match accuracy: 96.0%
Prefix match accuracy: 96.0%
Checkpoints saved to: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar_prime/caesar_prime_noisy_checkpoints/std_0p0
Wandb run: caesar_prime_noisy_std0p0_20260109_121806
